In [0]:
spark.sql("USE CATALOG catalog_project1")

DataFrame[]

In [0]:
# spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS").display()

databaseName
audit
bronze
default
gold
information_schema
schema_project1
silver
source1


In [0]:
spark.read.table("catalog_project1.silver.clean_products").display()
spark.read.table("catalog_project1.silver.clean_products").printSchema()

product_id,product_name,category,brand,price,created_date,modified_date,_ingestion_job_id,_source_table,_ingestion_pipeline_id,ingestion_timestamp,updation_timestamp
501,Laptop Pro 14,Electronics,Dell,85000.0,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
502,Gaming Mouse,Electronics,Logitech,2500.0,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
503,Mechanical Keyboard,Electronics,Keychron,6500.0,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
504,Office Chair,Furniture,Greensoul,12000.0,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
505,Study Table,Furniture,Ikea,15000.0,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
506,Water Bottle,Lifestyle,Milton,500.0,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
507,Wireless Earbuds,Electronics,Boat,3500.0,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
508,Smart Watch,Electronics,Noise,5500.0,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
509,Backpack,Lifestyle,Skybags,2200.0,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z,1784885093399,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-07-28T11:46:13.048Z
510,Monitor 27 Inch,Electronics,Lg,24000.0,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z,1785761612321,catalog_project1.bronze.products,74dbe2e5-6e5f-4be6-b843-e7992e743c8c,2026-07-28T11:46:13.048Z,2026-08-03T12:56:28.505Z


root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- modified_date: timestamp (nullable = true)
 |-- _ingestion_job_id: string (nullable = true)
 |-- _source_table: string (nullable = true)
 |-- _ingestion_pipeline_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- updation_timestamp: timestamp (nullable = true)



In [0]:
%skip 
dbutils.library.restartPython()

In [0]:
%skip 
import sys

# Add project root to sys.path so 'src' becomes importable
project_path = "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project"
if project_path not in sys.path:
    sys.path.insert(0, project_path) 

In [0]:
%skip 
import pyspark.sql.functions as F 
from pyspark.sql.window import Window
import uuid 
from delta.tables import DeltaTable 
from src.utils.audit import write_audit_log 
from datetime import datetime 
from src.utils.config import get_logger

start_time = datetime.now()

# pipeline id 
pipeline_id = str(uuid.uuid4())
scd1_column_list = ["product_name", "category", "brand"]
scd2_column_list = ["price"]
# last_product_key = 10 
error_message = None 
CATALOG = "catalog_project1"
SCHEMA = "gold"
pipeline_name = "gold_pipeline"
target = "dim_products"
AUDIT_SCHEMA = "audit"
AUDIT_TABLENAME = "pipeline_run_audit"
batch_id = str(uuid.uuid4())
watermark_column = "modified_date"
SOURCE_TABLE = "catalog_project1.silver.clean_products"
TARGET_TABLE = "catalog_project1.gold.dim_products"

# Fetch last execution 
# Fetch latest watermark value 

last_run = spark.table("catalog_project1.audit.pipeline_run_audit")\
    .filter((F.col("table_name") == "dim_products") & (F.col("status") == F.lit("SUCCESS")) & (F.col("pipeline_name") == F.lit("gold_pipeline")))\
    .orderBy(F.col("end_time").desc())\
    .limit(1)\
    .select("watermark_value").collect()
if last_run and last_run[0]['watermark_value'] is not None: 
    watermark_value = last_run[0]['watermark_value']
else: 
    watermark_value = None
# logger.info(f"Watermark value: {watermark_value}")

# Fetch records based on watermark value 
if watermark_value == None:  # Empty list or None
    silver_product = spark.table("catalog_project1.silver.clean_products")
else: 
    silver_product = spark.table("catalog_project1.silver.clean_products").filter(F.col("modified_date") > watermark_value)

rows_read = 0
rows_written = 0

try: 
    logger = get_logger(f"gold.{target}", log_to_file=True)
    logger.info(f"Starting Gold pipeline at {start_time}")

    rows_read = silver_product.count()
    logger.info(f"Read {rows_read} records from {SOURCE_TABLE} table")

    dim_product_table = "catalog_project1.gold.dim_products"

    if spark.catalog.tableExists(dim_product_table): 
        logger.info(f"Incremental load from {SOURCE_TABLE} table")
        dim_product = spark.table(dim_product_table).filter("is_current = true")

        # Fetch max product_key value 
        max_key = dim_product.agg(F.max("product_key")).collect()[0][0]
        max_key = max_key if max_key is not None else 0 
        # Generate new key window 
        window_spec = Window.orderBy(F.monotonically_increasing_id())

        # Compare with silver table records 
        comparision_df = (
            silver_product.alias('s')
            .join(
                dim_product.alias('d'), 
                on = F.expr("s.product_id = d.product_id"), 
                how = "left"
            )
        )

        # New records 
        new_products_df = comparision_df.filter(F.expr("d.product_id IS NULL"))
        new_records_count = new_products_df.count()
        if new_records_count > 0: 
            new_products_df = comparision_df.filter(F.expr("d.product_id IS NULL")).select("s.*") 
            new_products_df = new_products_df.withColumn("product_key", F.row_number().over(window_spec) + max_key)
            new_products_df = new_products_df.withColumns({
                "effective_from": F.current_timestamp(), 
                "effective_to": F.lit("9999-12-31"), 
                "is_current": F.lit(True),
                "ingestion_timestamp": F.current_timestamp(), 
                "updation_timestamp": F.current_timestamp(), 
                "_source_table": F.lit("catalog_project1.silver.clean_products"), 
                "_ingestion_pipeline_id": F.lit(pipeline_id) 
            })

            new_products_df.write.mode("append").format("delta").saveAsTable("catalog_project1.gold.dim_products")
            rows_written = new_records_count
            logger.info(f"Inserted {new_records_count} new records to {TARGET_TABLE} table")
        
        # Process SCD2 FIRST (price changes) - these take precedence
        # When SCD2 column changes, expire old record and insert new version with ALL current values
        scd2_df = comparision_df.filter(F.expr("d.product_id IS NOT NULL AND s.price != d.price"))
        scd2_count = scd2_df.count()
        if scd2_count > 0: 
            # Expire old records 
            expire_old = scd2_df.select("d.product_key").distinct()
            spark.sql(f"""
                      UPDATE {dim_product_table} 
                      SET is_current = false, 
                      effective_to = current_timestamp(), 
                      updation_timestamp = current_timestamp()
                      WHERE product_key IN ({','.join([str(r.product_key) for r in expire_old.collect()])})
            """) 
            # Insert new versions 
            new_versions = scd2_df.select("s.*")
            max_key = spark.table(dim_product_table).agg(F.max("product_key")).collect()[0][0]
            max_key = max_key if max_key is not None else 0
            window_spec = Window.orderBy(F.monotonically_increasing_id())
            new_versions = new_versions.withColumn("product_key", F.row_number().over(window_spec)  + max_key)\
            .withColumns({
                "effective_from": F.current_timestamp(), 
                "effective_to": F.lit("9999-12-31"), 
                "is_current": F.lit(True), 
                "ingestion_timestamp": F.current_timestamp(),
                "updation_timestamp": F.current_timestamp(),
                "_source_table": F.lit("catalog_project1.silver.clean_products"),
                "_ingestion_pipeline_id": F.lit(pipeline_id)
            })
            new_versions.write.mode("append").format("delta").saveAsTable(dim_product_table)
            rows_written += scd2_count
            logger.info(f"Inserted {scd2_count} new versions (SCD2) to {TARGET_TABLE} table")
        
        # Process SCD1 changes (product_name, category, brand) ONLY for records without SCD2 changes
        # Exclude records that already went through SCD2 processing
        scd2_product_ids = scd2_df.select("s.product_id").distinct() if scd2_count > 0 else spark.createDataFrame([], "product_id INT")
        
        scd1_df = comparision_df.filter(F.expr("d.product_id IS NOT NULL AND (NOT (s.product_name <=> d.product_name) OR NOT (s.category <=> d.category) OR NOT (s.brand <=> d.brand))"))
        scd1_count = scd1_df.count()
        if scd1_count > 0:
            # Exclude records that were processed in SCD2
            scd1_df = scd1_df.join(scd2_product_ids, on=F.expr("s.product_id = product_id"), how="left_anti")
            scd1_after_exclusion_count = scd1_df.count()
            
            if scd1_after_exclusion_count > 0:
                scd1_df = scd1_df.select(
                    F.col("s.product_id").alias("product_id"), 
                    F.col("s.product_name").alias("product_name"), 
                    F.col("s.category").alias("category"), 
                    F.col("s.brand").alias("brand"), 
                    F.col("s.modified_date").alias("modified_date")
                ).distinct()
                # Merge logic 
                dim_table = DeltaTable.forName(spark, dim_product_table)
                dim_table.alias("target")\
                    .merge(
                        scd1_df.alias("source"),
                        F.expr("target.product_id = source.product_id AND target.is_current = true")
                    )\
                    .whenMatchedUpdate(set = {
                        "product_name": F.col("source.product_name"),
                        "category": F.col("source.category"),
                        "brand": F.col("source.brand"),
                        "modified_date": F.col("source.modified_date"),
                        "updation_timestamp": F.current_timestamp()
                    })\
                    .execute()
                logger.info(f"Updated {scd1_after_exclusion_count} records (SCD1) in {TARGET_TABLE}")
        # Update watermark only if new data was processed
        if rows_written > 0:
            new_watermark = silver_product.agg(F.max(F.col(watermark_column))).collect()[0][0]
            logger.info(f"New watermark: {new_watermark}")
        else:
            new_watermark = watermark_value
            logger.info(f"No new data processed, watermark unchanged: {new_watermark}")
        status = "SUCCESS"
        logger.info(f"Incremental load completed for {TARGET_TABLE}")
        logger.info(f"Gold pipeline for {TARGET_TABLE} completed successfully")

    else: 
        # Initial load 
        logger.info(f"Initial load from {SOURCE_TABLE} table")
        # Surrogate key - product_key 
        df = silver_product.withColumn("product_key", F.monotonically_increasing_id() + 1) 

        # Metadata - effective_from, effective_to, is_current, ingestion_timestamp, updation_timestamp, source_table, pipeline_id 
        metadata = df.withColumns({
            "effective_from": F.current_timestamp(), 
            "effective_to": F.lit("9999-12-31"), 
            "is_current": F.lit(True), 
            "ingestion_timestamp": F.current_timestamp(), 
            "updation_timestamp": F.current_timestamp(),
            "_source_table": F.lit("catalog_project1.silver.clean_products"), 
            "_ingestion_pipeline_id": F.lit(pipeline_id)
        })

        # write to table 
        metadata.write.mode("overwrite").format("delta").saveAsTable("catalog_project1.gold.dim_products") 
        rows_written = spark.read.table("catalog_project1.gold.dim_products").count()
        if rows_written > 0:
            new_watermark = metadata.agg(F.max(F.col(watermark_column))).collect()[0][0]
            logger.info(f"New watermark: {new_watermark}")
        else: 
            new_watermark = watermark_value
            logger.info(f"No new data, watermark value unchanged: {new_watermark}")
        status = "SUCCESS"
        logger.info(f"Initial load completed for {TARGET_TABLE}")
        logger.info(f"Gold pipeline for {TARGET_TABLE} completed successfully")
    
except Exception as e: 
    status = "FAILURE"
    error_message = str(e)
    logger.exception(f"Gold pipeline failed: {error_message}")
    raise 

finally: 
    end_time = datetime.now()
    write_audit_log(pipeline_name, target, start_time, end_time, rows_read, rows_written, status, error_message, CATALOG, AUDIT_SCHEMA, AUDIT_TABLENAME, batch_id, watermark_column, new_watermark)
    logger.info(f"Audit for {TARGET_TABLE} written successfully")

# dim_prod = spark.read.table("catalog_project1.gold.dim_products")
# dim_prod.display()

2026-08-04 10:07:40,100 | INFO     | gold.dim_products | Logging to file: /Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/logs/gold_dim_products_20260804.log
2026-08-04 10:07:40,101 | INFO     | gold.dim_products | Starting Gold pipeline at 2026-08-04 10:07:39.190879
2026-08-04 10:07:40,482 | INFO     | gold.dim_products | Read 0 records from catalog_project1.silver.clean_products table
2026-08-04 10:07:40,781 | INFO     | gold.dim_products | Incremental load from catalog_project1.silver.clean_products table
2026-08-04 10:07:43,004 | INFO     | gold.dim_products | No new data processed, watermark unchanged: 2024-12-03 00:00:00
2026-08-04 10:07:43,005 | INFO     | gold.dim_products | Incremental load completed for catalog_project1.gold.dim_products
2026-08-04 10:07:43,006 | INFO     | gold.dim_products | Gold pipeline for catalog_project1.gold.dim_products completed successfully
2026-08-04 10:07:44,465 | INFO     | gold.dim_products | Audit fo

In [0]:
%skip 
spark.sql("DROP TABLE IF EXISTS catalog_project1.gold.dim_products")

DataFrame[]

In [0]:
spark.sql("SELECT * FROM catalog_project1.gold.dim_products").display()

product_id,product_name,category,brand,price,created_date,modified_date,_ingestion_job_id,_source_table,_ingestion_pipeline_id,ingestion_timestamp,updation_timestamp,product_key,effective_from,effective_to,is_current
501,Laptop Pro 14,Electronics,Dell,85000.0,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,1,2026-08-04T09:23:49.061Z,9999-12-31,true
502,Gaming Mouse,Electronics,Logitech,2500.0,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,2,2026-08-04T09:23:49.061Z,9999-12-31,true
503,Mechanical Keyboard,Electronics,Keychron,6500.0,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,3,2026-08-04T09:23:49.061Z,9999-12-31,true
504,Office Chair,Furniture,Greensoul,12000.0,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,4,2026-08-04T09:23:49.061Z,9999-12-31,true
505,Study Table,Furniture,Ikea,15000.0,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,5,2026-08-04T09:23:49.061Z,9999-12-31,true
506,Water Bottle,Lifestyle,Milton,500.0,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,6,2026-08-04T09:23:49.061Z,9999-12-31,true
507,Wireless Earbuds,Electronics,Boat,3500.0,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,7,2026-08-04T09:23:49.061Z,9999-12-31,true
508,Smart Watch,Electronics,Noise,5500.0,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,8,2026-08-04T09:23:49.061Z,9999-12-31,true
509,Backpack,Lifestyle,Skybags,2200.0,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,9,2026-08-04T09:23:49.061Z,9999-12-31,true
510,Monitor 27 Inch,Electronics,Lg,24000.0,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z,1785761612321,catalog_project1.silver.clean_products,c8977250-851f-4b4c-a184-1ea0469486ae,2026-08-04T09:23:49.061Z,2026-08-04T09:23:49.061Z,10,2026-08-04T09:23:49.061Z,9999-12-31,true


In [0]:
%skip 
spark.sql("SELECT * FROM catalog_project1.gold.dim_products WHERE updation_timestamp BETWEEN '2026-08-03T12:32:29.351+00:00' AND '2026-08-03T12:32:39.388+00:00'").display()

In [0]:
spark.sql("SELECT MAX(modified_date) FROM catalog_project1.gold.dim_products").display()

MAX(modified_date)
2024-12-03T00:00:00.000Z


In [0]:
spark.sql("SELECT * FROM catalog_project1.audit.pipeline_run_audit ORDER BY end_time DESC").display()

pipeline_name,table_name,start_time,end_time,rows_read,rows_written,status,error_message,batch_id,watermark_value,watermark_column
gold_pipeline,dim_products,2026-08-04T09:36:15.394Z,2026-08-04T09:36:29.047Z,2,0,SUCCESS,null,d891b48a-ead4-44b0-8710-2bdb27e4556b,2024-12-03T00:00:00.000Z,modified_date
silver_pipeline,clean_payments,2026-08-04T09:35:42.078Z,2026-08-04T09:35:50.226Z,0,0,SUCCESS,null,5ac3124f-4432-44d1-8235-21f82ad06408,2025-03-28T00:00:00.000Z,modified_date
silver_pipeline,clean_inventory,2026-08-04T09:35:32.562Z,2026-08-04T09:35:40.806Z,0,0,SUCCESS,null,591cf250-9aad-4486-958e-6bec9b67b443,2025-03-01T00:00:00.000Z,modified_date
silver_pipeline,clean_order_items,2026-08-04T09:35:22.546Z,2026-08-04T09:35:31.149Z,0,0,SUCCESS,null,811d664d-15a6-41b9-875a-8c08ed7973a7,2025-03-05T00:00:00.000Z,modified_date
silver_pipeline,clean_orders,2026-08-04T09:35:13.378Z,2026-08-04T09:35:21.152Z,0,0,SUCCESS,null,9667728f-147d-4ca6-b9bc-4c25525b7794,2025-03-28T00:00:00.000Z,modified_date
silver_pipeline,clean_products,2026-08-04T09:35:00.658Z,2026-08-04T09:35:11.988Z,2,2,SUCCESS,null,e0e16df7-13ee-404b-929f-225dc01cdecf,2024-12-03T00:00:00.000Z,modified_date
silver_pipeline,clean_customers,2026-08-04T09:34:49.141Z,2026-08-04T09:34:59.026Z,0,0,SUCCESS,null,0814c96d-10af-4298-b2e9-bb35e31b2dc0,2025-02-01T00:00:00.000Z,modified_date
bronze_ingestion_pipeline,order_items,2026-08-04T09:33:47.685Z,2026-08-04T09:33:52.471Z,0,0,SUCCESS,null,6374296c-8bdb-4957-99c7-5379811e7357,2025-03-05T00:00:00.000Z,modified_date
bronze_ingestion_pipeline,inventory,2026-08-04T09:33:41.270Z,2026-08-04T09:33:46.213Z,0,0,SUCCESS,null,6374296c-8bdb-4957-99c7-5379811e7357,2025-03-01T00:00:00.000Z,modified_date
bronze_ingestion_pipeline,payments,2026-08-04T09:33:33.713Z,2026-08-04T09:33:39.593Z,0,0,SUCCESS,null,6374296c-8bdb-4957-99c7-5379811e7357,2025-03-28T00:00:00.000Z,modified_date


In [0]:
%skip 
spark.sql("DELETE FROM catalog_project1.audit.pipeline_run_audit WHERE table_name = 'dim_products'")

DataFrame[num_affected_rows: bigint]

In [0]:
%skip 
spark.sql("SELECT * FROM catalog_project1.audit.pipeline_run_audit ORDER BY end_time DESC").display()

# After Code Refactoring 

In [0]:
import sys

# Add project root to sys.path so 'src' becomes importable
project_path = "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project"
if project_path not in sys.path:
    sys.path.insert(0, project_path) 

import pyspark.sql.functions as F 
from pyspark.sql.window import Window
import uuid 
from delta.tables import DeltaTable 
from src.utils.audit import write_audit_log 
from datetime import datetime 
from src.utils.config import get_logger, CATALOG, GOLD_SCHEMA, AUDIT_SCHEMA, AUDIT_TABLENAME, SILVER_SCHEMA, get_rows_count 
from src.reporting.gold import get_last_watermark, add_metadata, add_scd_metadata
from src.metadata.gold_pipeline_configuration import TABLE_CONFIG

# pipeline id, batch id 
pipeline_id = str(uuid.uuid4())
batch_id = str(uuid.uuid4())

metadata = TABLE_CONFIG["dim_products"]

scd1_column_list = metadata["scd1_columns"]
scd2_column_list = metadata["scd2_columns"]
error_message = None 
pipeline_name = "gold_pipeline"
source = metadata["source"]
target = metadata["target"]
watermark_column = metadata["watermark_column"]
rows_read = 0
rows_written = 0
rows_updated = 0
surrogate_key = metadata["surrogate_key"]
primary_key = metadata["primary_key"]
window_spec = Window.orderBy(F.monotonically_increasing_id())
new_watermark = None 

try: 
    start_time = datetime.now()
    logger = get_logger(f"gold.{target}", log_to_file=True)
    logger.info(f"Starting Gold pipeline at {start_time}")
    
    # Fetch latest watermark value 
    watermark_value = get_last_watermark(spark, CATALOG, AUDIT_SCHEMA, AUDIT_TABLENAME, target, pipeline_name)
    
    SOURCE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{source}"
    TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.{target}"

    # Fetch records based on watermark value 
    if watermark_value == None:  # Empty list or None
        silver_df = spark.table(SOURCE_TABLE)
    else: 
        silver_df = spark.table(SOURCE_TABLE).filter(F.col(watermark_column) > watermark_value)
    
    rows_read = get_rows_count(silver_df)
    logger.info(f"Read {rows_read} records from {SOURCE_TABLE} table")

    if spark.catalog.tableExists(TARGET_TABLE): 
        logger.info(f"Incremental load from {SOURCE_TABLE} table")
        # Get all current records for comparison
        existing_df = spark.table(TARGET_TABLE).filter("is_current = true")
        
        # Also get list of product_ids that have ANY version (current or expired)
        # to detect orphaned products that were expired but never got a new version
        all_product_ids = spark.table(TARGET_TABLE).select(primary_key).distinct()

        # Fetch max product_key value 
        max_key = existing_df.agg(F.max(surrogate_key)).collect()[0][0]
        max_key = max_key if max_key is not None else 0 
        # Generate new key window 
        # window_spec = Window.orderBy(F.monotonically_increasing_id())

        # Compare with silver table records 
        comparision_df = (
            silver_df.alias('s')
            .join(
                existing_df.alias('d'), 
                on = F.expr(f"s.{primary_key} = d.{primary_key}"), 
                how = "left"
            )
        )

        # New records (products that don't exist in Gold at all)
        truly_new_df = silver_df.alias('s').join(
            all_product_ids.alias('all'), 
            on=F.expr(f"s.{primary_key} = all.{primary_key}"), 
            how="left_anti"
        )
        truly_new_count = truly_new_df.count()
        if truly_new_count > 0:
            add_scd_metadata(truly_new_df, target, SOURCE_TABLE, pipeline_id, TARGET_TABLE, window_spec, max_key)
            rows_written = truly_new_count
            logger.info(f"Inserted {truly_new_count} new records to {TARGET_TABLE} table")
        
        # Orphaned records (products that exist but have no current version)
        # These need to be inserted as new current versions
        orphaned_df = comparision_df.filter(F.expr(f"d.{primary_key} IS NULL")).select("s.*")
        # Filter out the truly new ones - orphaned are those that exist in all_product_ids
        orphaned_df = orphaned_df.alias('orphan').join(
            all_product_ids.alias('all'),
            on=F.expr(f"orphan.{primary_key} = all.{primary_key}"),
            how="inner"
        ).select(f"orphan.*")
        orphaned_count = orphaned_df.count()
        if orphaned_count > 0:
            max_key = spark.table(TARGET_TABLE).agg(F.max(f"{surrogate_key}")).collect()[0][0]
            max_key = max_key if max_key is not None else 0
            add_scd_metadata(orphaned_df, target, SOURCE_TABLE, pipeline_id, TARGET_TABLE, window_spec, max_key)
            rows_written += orphaned_count
            logger.info(f"Inserted {orphaned_count} orphaned records (re-activated) to {TARGET_TABLE} table")
        
        # Process SCD2 FIRST (price changes) - these take precedence
        # When SCD2 column changes, expire old record and insert new version with ALL current values
        if scd2_column_list: 
            scd2_conditions = " OR ".join([f"s.{col} != d.{col}" for col in scd2_column_list])
            scd2_filter = f"d.{primary_key} IS NOT NULL AND ({scd2_conditions})"
            scd2_df = comparision_df.filter(F.expr(scd2_filter))
        else: 
            scd2_df = spark.createDataFrame([], schema=comparision_df.schema)
        scd2_count = scd2_df.count()
        if scd2_count > 0: 
            # Expire old records 
            expire_old = scd2_df.select(f"d.{surrogate_key}").distinct()
            expire_keys = [str(row[0]) for row in expire_old.collect()]
            spark.sql(f"""
                      UPDATE {TARGET_TABLE} 
                      SET is_current = false, 
                      effective_to = current_timestamp(), 
                      updation_timestamp = current_timestamp()
                      WHERE {surrogate_key} IN ({','.join(expire_keys)})
            """) 
            # Insert new versions 
            new_versions = scd2_df.select("s.*")
            max_key = spark.table(TARGET_TABLE).agg(F.max(f"{surrogate_key}")).collect()[0][0]
            max_key = max_key if max_key is not None else 0
            # window_spec = Window.orderBy(F.monotonically_increasing_id())
            add_scd_metadata(new_versions, target, SOURCE_TABLE, pipeline_id, TARGET_TABLE, window_spec, max_key)
            rows_written += scd2_count
            logger.info(f"Inserted {scd2_count} new versions (SCD2) to {TARGET_TABLE} table")
        
        # Process SCD1 changes (product_name, category, brand) ONLY for records without SCD2 changes
        # Exclude records that already went through SCD2 processing
        scd2_product_ids = scd2_df.select(f"s.{primary_key}").distinct() if scd2_count > 0 else spark.createDataFrame([], f"{primary_key} INT")
        
        if scd1_column_list: 
            scd1_conditions = " OR ".join([f"NOT (s.{col} <=> d.{col})" for col in scd1_column_list])
            scd1_filter = f"d.{primary_key} IS NOT NULL AND ({scd1_conditions})"
            scd1_df = comparision_df.filter(F.expr(scd1_filter))
        else: 
            scd1_df = spark.createDataFrame([], schema=comparision_df.schema)
        scd1_count = scd1_df.count()
        if scd1_count > 0:
            # Exclude records that were processed in SCD2
            scd1_df = scd1_df.join(scd2_product_ids, on=F.expr(f"s.{primary_key} = {primary_key}"), how="left_anti")
            scd1_after_exclusion_count = scd1_df.count()
            
            if scd1_after_exclusion_count > 0:
                select_cols = [primary_key] + scd1_column_list + [watermark_column]
                scd1_df = scd1_df.select(*[F.col(f"s.{col}").alias(col) for col in select_cols]).distinct()

                update_dict = {col: F.col(f"source.{col}") for col in scd1_column_list + [watermark_column]}
                update_dict["updation_timestamp"] = F.current_timestamp()

                # Merge logic 
                dim_table = DeltaTable.forName(spark, TARGET_TABLE)
                dim_table.alias("target")\
                    .merge(
                        scd1_df.alias("source"),
                        F.expr(f"target.{primary_key} = source.{primary_key} AND target.is_current = true")
                    )\
                    .whenMatchedUpdate(set = update_dict)\
                    .execute()
                rows_updated += scd1_after_exclusion_count
                logger.info(f"Updated {scd1_after_exclusion_count} records (SCD1) in {TARGET_TABLE}")
        # Update watermark if ANY data was processed (inserts or updates)
        if rows_written > 0 or rows_updated > 0:
            new_watermark = silver_df.agg(F.max(F.col(watermark_column))).collect()[0][0]
            logger.info(f"New watermark: {new_watermark}")
        else:
            new_watermark = watermark_value
            logger.info(f"No new data processed, watermark unchanged: {new_watermark}")
        status = "SUCCESS"
        logger.info(f"Incremental load completed for {TARGET_TABLE}")
        logger.info(f"Gold pipeline for {TARGET_TABLE} completed successfully")

    else: 
        # Initial load 
        logger.info(f"Initial load from {SOURCE_TABLE} table")
        add_metadata(silver_df, target, SOURCE_TABLE, pipeline_id, TARGET_TABLE)
        rows_written = silver_df.count()
        logger.info(f"Written {rows_written} records to {TARGET_TABLE}")
        if rows_written > 0:
            new_watermark = silver_df.agg(F.max(F.col(watermark_column))).collect()[0][0]
            logger.info(f"New watermark: {new_watermark}")
        else: 
            new_watermark = watermark_value
            logger.info(f"No new data, watermark value unchanged: {new_watermark}")
        status = "SUCCESS"
        logger.info(f"Initial load completed for {TARGET_TABLE}")
        logger.info(f"Gold pipeline for {TARGET_TABLE} completed successfully")
    
except Exception as e: 
    status = "FAILURE"
    error_message = str(e)
    logger.exception(f"{pipeline_name} failed: {error_message}")
    raise 

finally: 
    end_time = datetime.now()
    try: 
        write_audit_log(pipeline_name, target, start_time, end_time, rows_read, rows_written, status, error_message, CATALOG, AUDIT_SCHEMA, AUDIT_TABLENAME, batch_id, watermark_column, new_watermark)
        logger.info(f"Audit for {TARGET_TABLE} is written successfully")
    except Exception as e: 
        logger.exception(f"Error writing audit log for {TARGET_TABLE}: {e}")
        raise
 
# dim_prod = spark.read.table("catalog_project1.gold.dim_products")
# dim_prod.display() 

2026-08-04 13:52:48,646 | INFO     | gold.dim_products | Logging to file: /Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/logs/gold_dim_products_20260804.log
2026-08-04 13:52:48,647 | INFO     | gold.dim_products | Starting Gold pipeline at 2026-08-04 13:52:48.613305
2026-08-04 13:52:49,873 | INFO     | gold.dim_products | Read 0 records from catalog_project1.silver.clean_products table
2026-08-04 13:52:50,217 | INFO     | gold.dim_products | Incremental load from catalog_project1.silver.clean_products table
2026-08-04 13:52:53,537 | INFO     | gold.dim_products | No new data processed, watermark unchanged: 2024-12-04 00:00:00
2026-08-04 13:52:53,538 | INFO     | gold.dim_products | Incremental load completed for catalog_project1.gold.dim_products
2026-08-04 13:52:53,539 | INFO     | gold.dim_products | Gold pipeline for catalog_project1.gold.dim_products completed successfully
2026-08-04 13:52:55,328 | INFO     | gold.dim_products | Audit fo

In [0]:
spark.sql("SELECT * FROM catalog_project1.gold.dim_products").display()

product_id,product_name,category,brand,price,created_date,modified_date,_ingestion_job_id,_source_table,_ingestion_pipeline_id,ingestion_timestamp,updation_timestamp,product_key,effective_from,effective_to,is_current
501,Laptop Pro 14,Electronics,Dell,85000.0,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,1,2026-08-04T12:40:52.750Z,9999-12-31,true
502,Gaming Mouse,Electronics,Logitech,2500.0,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,2,2026-08-04T12:40:52.750Z,9999-12-31,true
503,Mechanical Keyboard,Electronics,Keychron,6500.0,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,3,2026-08-04T12:40:52.750Z,9999-12-31,true
504,Office Chair,Furniture,Greensoul,12000.0,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,4,2026-08-04T12:40:52.750Z,9999-12-31,true
505,Study Table,Furniture,Ikea,15000.0,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,5,2026-08-04T12:40:52.750Z,9999-12-31,true
506,Water Bottle,Lifestyle,Milton,500.0,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,6,2026-08-04T12:40:52.750Z,9999-12-31,true
507,Wireless Earbuds,Electronics,Boat,3500.0,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,7,2026-08-04T12:40:52.750Z,9999-12-31,true
508,Smart Watch,Electronics,Noise,5500.0,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,8,2026-08-04T12:40:52.750Z,9999-12-31,true
509,Backpack,Lifestyle,Skybags,2200.0,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z,1784885093399,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,9,2026-08-04T12:40:52.750Z,9999-12-31,true
510,Monitor 27 Inch,Electronics,Lg,24000.0,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z,1785761612321,catalog_project1.silver.clean_products,b0429de9-729b-4153-be7d-f9bd6dd54b22,2026-08-04T12:40:52.750Z,2026-08-04T12:40:52.750Z,10,2026-08-04T12:40:52.750Z,9999-12-31,true


In [0]:
spark.sql("SELECT * FROM catalog_project1.audit.pipeline_run_audit ORDER BY end_time DESC").display()

pipeline_name table_name start_time end_time rows_read rows_written status error_message batch_id watermark_value watermark_column gold_pipeline dim_products 2026-08-04T13:52:48.613Z 2026-08-04T13:52:53.539Z 0 0 SUCCESS null 579699b3-46e7-4bd0-8816-ffc9ff2f2ce2 2024-12-04T00:00:00.000Z modified_date gold_pipeline dim_products 2026-08-04T13:47:57.166Z 2026-08-04T13:48:05.447Z 2 1 SUCCESS null 2e0278ff-21f9-4072-908a-58758046645d 2024-12-04T00:00:00.000Z modified_date gold_pipeline dim_products 2026-08-04T13:45:32.458Z 2026-08-04T13:45:35.901Z 0 0 FAILURE [AMBIGUOUS_REFERENCE] Reference `orphan`.`product_id` is ambiguous, could be: [`orphan`.`product_id`, `orphan`.`product_id`]. SQLSTATE: 42704; line 1 pos 0

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:3783)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:372)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:266)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpressionByPlanChildren$1(ColumnResolutionHelper.scala:495)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$3(ColumnResolutionHelper.scala:156)
	at org.apache.spark.sql.catalyst.analysis.package$.withPosition(package.scala:110)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$1(ColumnResolutionHelper.scala:166)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:142)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.innerResolve$1(ColumnResolutionHelper.scala:121)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$9(ColumnResolutionHelper.scala:200)
	at org.apache.spark.sql.catalyst.trees.BinaryLike.mapChildren(TreeNode.scala:1500)
	at org.apache.spark.sql.catalyst.trees.BinaryLike.mapChildren$(TreeNode.scala:1499)
	at org.apache.spark.sql.catalyst.expressions.BinaryExpression.mapChildren(Expression.scala:923)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$1(ColumnResolutionHelper.scala:200)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:142)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.innerResolve$1(ColumnResolutionHelper.scala:121)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpression(ColumnResolutionHelper.scala:207)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpression$(ColumnResolutionHelper.scala:112)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveReferences.resolveExpression(Analyzer.scala:2358)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpressionByPlanChildren(ColumnResolutionHelper.scala:502)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpressionByPlanChildren$(ColumnResolutionHelper.scala:487)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveReferences.resolveExpressionByPlanChildren(Analyzer.scala:2358)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveReferences$$anonfun$doApply$3.$anonfun$applyOrElse$110(Analyzer.scala:2741)
	at org.apache.spark.sql.catalyst.plans.QueryPlan.$anonfun$mapExpressions$1(QueryPlan.scala:266)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:142)
	at org.apache.spark.sql.catalyst.plans.QueryPlan.transformExpression$1(QueryPlan.scala:266)
	at org.apache.spark.sql.catalyst.plans.QueryPlan.recursiveTransform$1(QueryPlan.scala:278)
	at org.apache.spark.sql.catalyst.plans.QueryPlan.recursiveTransform$1(QueryPlan.scala:279)
	at org.apache.spark.sql.catalyst.plans.QueryPlan.$anonfun$mapExpressions$5(QueryPlan.scala:289)
	at org.apache.spark.sql.catalyst.trees.TreeNode.mapProductIterator(TreeNode.scala:452)
	

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys 

# Add project root to sys.path so 'src' becomes importable
project_path = "/Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project"
if project_path not in sys.path:
    sys.path.insert(0, project_path) 

import src.reporting.gold 
from src.reporting.gold import create_gold_table
from src.metadata.gold_pipeline_configuration import TABLE_CONFIG 

for table, metadata in TABLE_CONFIG.items(): 
    create_gold_table(scd1_column_list = metadata["scd1_columns"], 
                      scd2_column_list = metadata["scd2_columns"], 
                      source = metadata["source"], 
                      target = metadata["target"], 
                      watermark_column = metadata["watermark_column"], 
                      surrogate_key = metadata["surrogate_key"], 
                      primary_key = metadata["primary_key"])

2026-08-05 09:49:45,727 | INFO     | gold.dim_products | Logging to file: /Workspace/Users/krishnasumanbhaipatel@gmail.com/Practice Folder/lakehouse-practice-project/logs/gold_dim_products_20260805.log
2026-08-05 09:49:45,727 | INFO     | gold.dim_products | Starting Gold pipeline at 2026-08-05 09:49:45.691839
2026-08-05 09:49:47,267 | INFO     | gold.dim_products | Read 0 records from catalog_project1.silver.clean_products table
2026-08-05 09:49:47,494 | INFO     | gold.dim_products | Incremental load from catalog_project1.silver.clean_products table
2026-08-05 09:49:51,894 | INFO     | gold.dim_products | No new data processed, watermark unchanged: 2024-12-04 00:00:00
2026-08-05 09:49:51,895 | INFO     | gold.dim_products | Incremental load completed for catalog_project1.gold.dim_products
2026-08-05 09:49:51,895 | INFO     | gold.dim_products | Gold pipeline for catalog_project1.gold.dim_products completed successfully
2026-08-05 09:49:53,969 | INFO     | gold.dim_products | Audit fo

In [0]:
spark.sql("SELECT * FROM catalog_project1.gold.dim_customers").display()

customer_id,name,email,city,state,signup_date,created_date,modified_date,_ingestion_job_id,_source_table,_ingestion_pipeline_id,ingestion_timestamp,updation_timestamp,product_key,effective_from,effective_to,is_current
1001,Rahul Shah,rahul.shah@email.com,Ahmedabad,Gujarat,2025-01-10T00:00:00.000Z,2025-01-10T00:00:00.000Z,2025-01-10T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,1,2026-08-05T09:49:58.283Z,9999-12-31,true
1002,Priya Patel,priya.patel@email.com,Surat,Gujarat,2025-01-12T00:00:00.000Z,2025-01-12T00:00:00.000Z,2025-01-12T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,2,2026-08-05T09:49:58.283Z,9999-12-31,true
1003,Amit Verma,amit.verma@email.com,Delhi,Delhi,2025-01-15T00:00:00.000Z,2025-01-15T00:00:00.000Z,2025-01-15T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,3,2026-08-05T09:49:58.283Z,9999-12-31,true
1004,Sneha Iyer,sneha.iyer@email.com,Bengaluru,Karnataka,2025-01-18T00:00:00.000Z,2025-01-18T00:00:00.000Z,2025-01-18T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,4,2026-08-05T09:49:58.283Z,9999-12-31,true
1005,Rohan Gupta,rohan.gupta@email.com,Mumbai,Maharashtra,2025-01-20T00:00:00.000Z,2025-01-20T00:00:00.000Z,2025-01-20T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,5,2026-08-05T09:49:58.283Z,9999-12-31,true
1006,Neha Sharma,neha.sharma@email.com,Jaipur,Rajasthan,2025-01-22T00:00:00.000Z,2025-01-22T00:00:00.000Z,2025-01-22T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,6,2026-08-05T09:49:58.283Z,9999-12-31,true
1007,Karan Mehta,karan.mehta@email.com,Pune,Maharashtra,2025-01-24T00:00:00.000Z,2025-01-24T00:00:00.000Z,2025-01-24T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,7,2026-08-05T09:49:58.283Z,9999-12-31,true
1008,Anjali Singh,anjali.singh@email.com,Lucknow,Uttar Pradesh,2025-01-27T00:00:00.000Z,2025-01-27T00:00:00.000Z,2025-01-27T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,8,2026-08-05T09:49:58.283Z,9999-12-31,true
1009,Vikram Joshi,vikram.joshi@email.com,Indore,Madhya Pradesh,2025-01-29T00:00:00.000Z,2025-01-29T00:00:00.000Z,2025-01-29T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,9,2026-08-05T09:49:58.283Z,9999-12-31,true
1010,Meera Nair,meera.nair@email.com,Kochi,Kerala,2025-02-01T00:00:00.000Z,2025-02-01T00:00:00.000Z,2025-02-01T00:00:00.000Z,1784885085539,catalog_project1.silver.clean_customers,e3a13bbf-c60c-4d81-91e3-fc192f3326f2,2026-08-05T09:49:58.283Z,2026-08-05T09:49:58.283Z,10,2026-08-05T09:49:58.283Z,9999-12-31,true


In [0]:
spark.sql("SELECT * FROM catalog_project1.audit.pipeline_run_audit ORDER BY end_time DESC").display()

pipeline_name table_name start_time end_time rows_read rows_written status error_message batch_id watermark_value watermark_column gold_pipeline dim_customers 2026-08-05T09:49:53.970Z 2026-08-05T09:50:00.841Z 10 10 SUCCESS null bb8b13ce-7c21-4e99-a4f0-00504257033e 2025-02-01T00:00:00.000Z modified_date gold_pipeline dim_products 2026-08-05T09:49:45.691Z 2026-08-05T09:49:51.896Z 0 0 SUCCESS null c7a51f34-ee35-4a2f-95c4-d0892152fd4c 2024-12-04T00:00:00.000Z modified_date gold_pipeline dim_products 2026-08-05T09:39:53.400Z 2026-08-05T09:40:09.835Z 0 0 SUCCESS null 4d771246-d1a9-4dee-9408-42db13d6c762 2024-12-04T00:00:00.000Z modified_date gold_pipeline dim_products 2026-08-04T13:52:48.613Z 2026-08-04T13:52:53.539Z 0 0 SUCCESS null 579699b3-46e7-4bd0-8816-ffc9ff2f2ce2 2024-12-04T00:00:00.000Z modified_date gold_pipeline dim_products 2026-08-04T13:47:57.166Z 2026-08-04T13:48:05.447Z 2 1 SUCCESS null 2e0278ff-21f9-4072-908a-58758046645d 2024-12-04T00:00:00.000Z modified_date gold_pipeline dim_products 2026-08-04T13:45:32.458Z 2026-08-04T13:45:35.901Z 0 0 FAILURE [AMBIGUOUS_REFERENCE] Reference `orphan`.`product_id` is ambiguous, could be: [`orphan`.`product_id`, `orphan`.`product_id`]. SQLSTATE: 42704; line 1 pos 0

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:3783)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:372)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:266)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpressionByPlanChildren$1(ColumnResolutionHelper.scala:495)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$3(ColumnResolutionHelper.scala:156)
	at org.apache.spark.sql.catalyst.analysis.package$.withPosition(package.scala:110)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$1(ColumnResolutionHelper.scala:166)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:142)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.innerResolve$1(ColumnResolutionHelper.scala:121)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$9(ColumnResolutionHelper.scala:200)
	at org.apache.spark.sql.catalyst.trees.BinaryLike.mapChildren(TreeNode.scala:1500)
	at org.apache.spark.sql.catalyst.trees.BinaryLike.mapChildren$(TreeNode.scala:1499)
	at org.apache.spark.sql.catalyst.expressions.BinaryExpression.mapChildren(Expression.scala:923)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpression$1(ColumnResolutionHelper.scala:200)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:142)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.innerResolve$1(ColumnResolutionHelper.scala:121)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpression(ColumnResolutionHelper.scala:207)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpression$(ColumnResolutionHelper.scala:112)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveReferences.resolveExpression(Analyzer.scala:2358)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpressionByPlanChildren(ColumnResolutionHelper.scala:502)
	at org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.resolveExpressionByPlanChildren$(ColumnResolutionHelper.scala:487)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveReferences.resolveExpressionByPlanChildren(Analyzer.scala:2358)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveReferences$$anonfun$doApply$3.$anonfun$applyOrElse$110(Analyzer.scala:2741)
	at org.apache.spark.sql.catalyst.plans.QueryPlan.$anonfun$mapExpressions$1(QueryPlan.scala:266)
	at org.apache.spark.sql.catalyst.

In [0]:
# Enable column mapping first (required for column rename in Delta)
spark.sql("""
    ALTER TABLE catalog_project1.gold.dim_customers 
    SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

# Now rename the column
spark.sql("""
    ALTER TABLE catalog_project1.gold.dim_customers 
    RENAME COLUMN product_key TO customer_key
""")

DataFrame[]